In [2]:
import numpy as np
import pandas as pd
import sys
from collections import namedtuple
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import os
from pathlib import Path
import torch
import torch.nn as nn
from gym import spaces
from maddpg_agent_v2_test import Agent
import time
from collections import defaultdict

In [8]:
class MultiAgentEnergyStorageEnv:
    def __init__(self,
                 data_path: str = None,
                 num_agents: int = 3,
                 episode_length: int =96,
                 battery_capacity: float = 2,
                 max_charge_rate: float = 1,
                 efficiency: float = 0.95,
                 init_soc: float = 0.5,
                 seed: int = 42):
        self.num_agents = num_agents
        self.episode_length = episode_length
        self.battery_capacity = battery_capacity
        self.max_charge_rate = max_charge_rate
        self.efficiency = efficiency
        self.init_soc = init_soc
        self.seed = seed
        self.charge_power = None
        np.random.seed(seed)

        if data_path is None:
            if 'ipykernel' in sys.modules:  # Jupyter
                data_path = Path.cwd().parent / "data" / "opsd_building.csv"
            else:  # Normal Python
                data_path = Path(__file__).parent.parent / "data" / "opsd_building.csv"

        # load
        self.raw_data = self._load_data(data_path)
        self.load_profiles = self._process_load_data()

        # check data
        self._validate_data()
        
        # Initialization state
        self.current_step = 0
        self.battery_soc = {agent_id: init_soc for agent_id in range(num_agents)}
        self.charge_history = {agent_id: [] for agent_id in range(num_agents)}
        self.cost_history = {agent_id: [] for agent_id in range(num_agents)}
        self.load_history = {agent_id: [] for agent_id in range(num_agents)}
        self.price_history = []
        self.net_load_history = []

    def _load_data(self, data_path: str) -> pd.DataFrame:
        """from csv file"""
        try:
            data = pd.read_csv(data_path)
            print("Data loaded successfully, first 5 rows example:")
            print(data.head())
            return data
        except Exception as e:
            raise ValueError(f"Unable to load data file: {e}")
        
    def _validate_data(self):
        """Verify data integrity and length"""
        # 检查负荷数据
        for agent_id, profile in self.load_profiles.items():
            if len(profile) < self.episode_length:
                raise ValueError(f"The payload data length is insufficient and needs to be at least{self.episode_length}time points")

        print(f"data check pass,A total of {len(self.raw_data)}row of data is loaded")

    def _process_load_data(self) -> Dict[int, pd.DataFrame]:
        """Process load data and convert it into load curve for each agent"""
        load_columns = [f"load{i+1}" for i in range(self.num_agents)]
        # check if the required load columns exist in the raw data
        missing_cols = [col for col in load_columns if col not in self.raw_data.columns]
        if missing_cols:
            raise ValueError(f"Load column missing from the data file: {missing_cols}")
        
        num_rows = self.raw_data.shape[0]
        num_days = num_rows // self.episode_length

        # Load data columns (columns 2-4 are loads)
        load_data = self.raw_data.iloc[:, 1:4].values  # shape: (num_rows, 3)
        price_data = self.raw_data.iloc[:, -1].values  # last column is the electricity price

        load_profiles = {}
        day_idx = np.random.randint(0, num_days)
        for agent_id in range(self.num_agents):
            # Randomly sample the load curve for a day, but the sampling period is the same for all agents
            start = day_idx * self.episode_length
            end = start + self.episode_length
            agent_load = load_data[start:end, agent_id]
            agent_price = price_data[start:end]

            df = pd.DataFrame({
            'period': np.arange(self.episode_length),
            'load': agent_load,
            'price': agent_price
            })
            load_profiles[agent_id] = df

        # Note: The DataFrame returned here for each agent contains the load and electricity price
        return load_profiles


    def action_space(self, agent_name=None):
        """
        返回单个agent或所有agent的动作空间
        参数:
            agent_name: 可选，指定agent的ID或名称
        返回:
            如果指定agent_name: 返回该agent的gym.Space对象
            如果未指定: 返回所有agent的动作空间字典
        """
        # 定义单个agent的动作空间（连续功率值）
        single_action_space = spaces.Box(
            low=-self.max_charge_rate,
            high=self.max_charge_rate,
            shape=(1,),
            dtype=np.float32
        )
        
        if agent_name is not None:
            # 返回指定agent的空间
            return single_action_space
        else:
            # 返回所有agent的空间字典
            return {
                agent_id: single_action_space 
                for agent_id in range(self.num_agents)
            }
    
    def _get_single_observation(self,agent_id: int) -> np.ndarray:
        """定义每个智能体的观测空间"""
        angle = 2 * np.pi * self.current_step / 96
        current_price = self.load_profiles[agent_id]['price'].iloc[self.current_step]  #/163.52 # Normalized electricity price data
        current_load = self.load_profiles[agent_id]['load'].iloc[self.current_step] #/ 1.663  # Normalized load data
        current_soc = float(self.battery_soc[agent_id])
        return np.array([np.sin(angle),np.cos(angle), current_price, current_load, current_soc])
 
    def reset(self) -> Dict[int, np.ndarray]:
        """
        重置环境状态
        返回:
            每个智能体的初始观测
        """
        self.current_step = 0
        self.battery_soc = {agent_id: self.init_soc for agent_id in range(self.num_agents)}
        self.charge_history = {agent_id: [] for agent_id in range(self.num_agents)}
        self.load_history = {agent_id: [] for agent_id in range(self.num_agents)}
        self.price_history = []
        self.net_load_history = []
        
        # 获取初始观测
        observations = {}
        for agent_id in range(self.num_agents):
            observations[agent_id] = self._get_single_observation(agent_id)
        
        return observations
    
    def step(self, actions: Dict[int, float]) -> Tuple[Dict[int, np.ndarray], Dict[int, float], bool, Dict]:
        """
        Executes one time step of environmental dynamics
        Parameters:
        actions: dictionary format {agent_id: charging power}, positive value is charging, negative value is discharging
        Returns:
        observations: new observations of each agent
        rewards: rewards for each agent
        done: whether to terminate
        info: additional information (such as market settlement results)
        """
        # 1. Save current load and electricity price status
        current_loads = {
            agent_id: self.load_profiles[agent_id]['load'].iloc[self.current_step]
            for agent_id in range(self.num_agents)
        }
        current_price = self.load_profiles[0]['price'].iloc[self.current_step]  # 当前所有agent电价相同

        # Implementing battery dynamics (core physics model)
        net_loads = {}
        individual_costs = {}
        delta_soc=0
        for agent_id, action in actions.items():
            # Limit actions to the allowed range
            charge_power = np.clip(action, -self.max_charge_rate, self.max_charge_rate)
            if charge_power >= 0:
                delta_soc = (charge_power * self.efficiency) / self.battery_capacity
            elif charge_power < 0:
                delta_soc = (charge_power / self.efficiency) / self.battery_capacity
            new_soc = self.battery_soc[agent_id]+delta_soc

            #Bounds Checking and Penalization
            if new_soc > 1.0:  # over charge
                new_soc = 1.0
                charge_power = (1.0 - self.battery_soc[agent_id]) * self.battery_capacity / self.efficiency
            elif new_soc < 0.0:  # over discharge
                new_soc = 0.0
                charge_power = -self.battery_soc[agent_id] * self.battery_capacity * self.efficiency
            # update battery state
            self.battery_soc[agent_id] = new_soc
            self.charge_history[agent_id].append(charge_power)

            # Calculate the net load (actual load - discharge + charge)
            net_load = float(current_loads[agent_id] - charge_power)  # 放电为正，充电为负
            net_loads[agent_id] = net_load
            
            # Calculate individual costs
            individual_costs[agent_id] = net_load * current_price

            # Record the cost at each time step (for final calculations)
            self.cost_history[agent_id].append(net_load * current_price)

            #record
            self.charge_history[agent_id].append(charge_power)
            self.load_history[agent_id].append(current_loads[agent_id])
            self.price_history.append(current_price)

        # calculate global net load
        average_net_load = sum(net_loads.values())/ self.episode_length
        self.net_load_history.append(average_net_load)

        # get new observations
        next_observations = {}
        for agent_id in range(self.num_agents):
            angle = 2 * np.pi * (self.current_step+1) / 96
            next_observations[agent_id] = np.array([
                np.sin(angle),np.cos(angle),
                self.load_profiles[agent_id]['price'].iloc[self.current_step+1],# / 163.52,
                self.load_profiles[agent_id]['load'].iloc[self.current_step+1] /1.663,
                float(self.battery_soc[agent_id])
            ], dtype=np.float32)

        # reward
        rewards = self._calculate_rewards(individual_costs, average_net_load)

        # check if the episode is done
        self.current_step += 1
        if self.current_step >= self.episode_length-1:
            done = {agent_id: True for agent_id in range(self.num_agents)}
        else:
            done = {agent_id: False for agent_id in range(self.num_agents)}

        info = {
            'average_net_load': average_net_load,
            'average_cost': np.mean(list(individual_costs.values())),
            'peak_load': max(self.net_load_history) if self.net_load_history else 0,
            'valley_load': min(self.net_load_history) if self.net_load_history else 0,
            'current_period': self.current_step
        }
        
        return next_observations, rewards, done, done, info

    # def _calculate_daily_rewards(self) -> Dict[int, float]:
    #     """计算基于日总成本的奖励（负成本=正奖励）"""
    #     daily_rewards = {}
    #     for agent_id in range(self.num_agents):
    #         # 1. 计算日总电费
    #         total_cost = sum(self.cost_history[agent_id])  # 所有时间步成本求和
            
    #         # 3. 组合奖励（最大化负成本）
    #         daily_rewards[agent_id] = -total_cost  # 单位：元
            
    #         # 4. 可选：按比例缩放奖励数值（示例缩放至[-10,10]范围）
    #         daily_rewards[agent_id] /= 1000  
        
    #     return daily_rewards


    def _calculate_rewards(self, individual_costs: Dict[int, float], average_net_load: float) -> Dict[int, float]:
        """Calculate rewards combining individual cost optimization and load curve flattening.
        
        Args:
            individual_costs: Dictionary mapping agent IDs to their electricity costs
            average_net_load: Current average net load across all agents
            
        Returns:
            Dictionary mapping agent IDs to their calculated rewards
        """
        # 1. Individual reward component: Minimize electricity cost (negative cost)
        individual_rewards = {
            agent_id: float(-cost) 
            for agent_id, cost in individual_costs.items()
        }

        # 2. Global reward component: Load curve flattening
        # Calculate peak and valley thresholds from historical data
        peak_threshold = max(self.net_load_history) if self.net_load_history else 0
        valley_threshold = min(self.net_load_history) if self.net_load_history else 0
        
        # Calculate peak/valley penalties (squared to emphasize large deviations)
        peak_penalty = max(0, average_net_load - peak_threshold) ** 2
        valley_penalty = max(0, valley_threshold - average_net_load) ** 2
        netload_penalty = (peak_penalty + valley_penalty) / 2  # Average penalty
        netload_reward = -netload_penalty  # Convert penalty to negative reward

        # 3. Combine reward components
        rewards = {}
        for agent_id in individual_rewards.keys():
            # Weighted sum of components (weights can be tuned):
            # - 50% individual cost optimization
            # - 15% load flattening (30% of 50% to temporarily reduce impact)
            rewards[agent_id] = (
                0.5 * individual_rewards[agent_id] + 
                0.15 * netload_reward / self.num_agents  # Distributed equally
            )

        return rewards

    def close(self):
        pass
        print("env closed")

In [9]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
NUM_EPISODE = 100
NUM_STEP = 96
NUM_AGENT = 3
WARMUP_EPISODES = 500

LR_ACTOR = 0.001
LR_CRITIC = 0.001
HIDDEN_DIM = 32
GAMMA = 0.99
TAU = 0.01
MEMORY_SIZE = 5000
BATCH_SIZE = 256
TARGET_UPDATE_INTERVAL = 100

PRINT_INTERVAL = 50
highest_reward = 0

Using device:  cpu


In [10]:
#1. initialize
def multi_obs_to_state(multi_obs):
    state = np.array([])
    for agent_obs in multi_obs.values():
        state = np.concatenate([state,agent_obs])
    return state
env = MultiAgentEnergyStorageEnv(num_agents=NUM_AGENT, episode_length=NUM_STEP)

Data loaded successfully, first 5 rows example:
     unixtime  load1  load2  load3  price
0  1432223100  0.079  0.187  0.039  29.93
1  1432224000  0.078  0.000  0.031  29.93
2  1432224900  0.080  0.000  0.075  29.62
3  1432225800  0.110  0.000  0.176  29.62
4  1432226700  0.107  0.135  0.541  29.62
data check pass,A total of 63456row of data is loaded


In [13]:
multi_obs = env.reset()
agent_name_list = list(multi_obs.keys())
# print("basic observation:", agent_name_list)
#1.1 observation dimension
obs_dim = []
for agent_obs in multi_obs.values():
    obs_dim.append(agent_obs.shape[0])
state_dim = sum(obs_dim)

#1.2action dimension
action_dim = []
for agent_name in agent_name_list:
    action_dim.append(env.action_space(agent_name).sample().shape[0])


agents=[]
for agent_i in range(NUM_AGENT):
    print(f'Initialize the Agent{agent_i}...')
    agent =Agent(memo_size=MEMORY_SIZE, obs_dim=obs_dim[agent_i], state_dim=state_dim, n_agent=NUM_AGENT, 
                 action_dim=action_dim[agent_i], alpha=LR_ACTOR, beta=LR_CRITIC, fc1_dims=HIDDEN_DIM, fc2_dims=HIDDEN_DIM, gamma=GAMMA, tau=TAU, batch_size=BATCH_SIZE)
    
    agents.append(agent)

#保存模型路径
scenario = "buliding_test"
print(f"Scenario: {scenario}")
current_path = os.getcwd()
agent_path = current_path + '/models/' + scenario + '/'
timestamp = time.strftime("%Y%m%d%H%M%S")

Initialize the Agent0...
Initialize the Agent1...
Initialize the Agent2...
Scenario: buliding_test


In [ ]:
#2.train
EPISODE_REWARD_BUFFER = []
AVG_EPISODE_REWARD_BUFFER = []
agent_data = {
    agent_name: {
        'actions': defaultdict(list),
        'obs': defaultdict(list)
    } for agent_name in agent_name_list
}
for episode_i in range(NUM_EPISODE):
    multi_obs = env.reset()
    episode_reward = 0
    multi_done = {agent_name: False for agent_name in agent_name_list}

    # initialize current episode data
    current_episode_actions = {agent_name: [] for agent_name in agent_name_list}
    current_episode_obs = {agent_name: [] for agent_name in agent_name_list}

    # record the first observation
    for agent_name in agent_name_list:
        current_episode_obs[agent_name].append(multi_obs[agent_name].tolist())
        # current_episode_actions[agent_name].append([None]*len(multi_action[agent_name]))
    for agent_name in agent_name_list:
        current_episode_obs[agent_name].append(multi_obs[agent_name].tolist())

    for step_i in range(NUM_STEP):
        #if done
        if any(multi_done.values()):
            break
        total_step = episode_i * NUM_STEP + step_i
        print('total_step:', total_step, end='\r')

        # 收集每个智能体的动作
        multi_action={}
        for agent_i, agent_name in enumerate(agent_name_list):
            #  choose action
            agent = agents[agent_i]
            single_obs = multi_obs[agent_name]
            # print(f"Agent{agent_name}'s observation: {single_obs}")
            single_action = agent.get_action(single_obs)
            multi_action[agent_name]=single_action

        # Execute an action
        multi_next_obs, multi_reward, multi_done, multi_truncations, infos = env.step(multi_action)
        state = multi_obs_to_state(multi_obs)
        next_state = multi_obs_to_state(multi_next_obs)

        if step_i >= NUM_STEP -1 :
            multi_done = {agent_name: True for agent_name in agent_name_list}

        # save
        for agent_i, agent_name in enumerate(agent_name_list):
            agent = agents[agent_i]
            single_obs = multi_obs[agent_name]
            single_next_obs = multi_next_obs[agent_name]
            single_action = multi_action[agent_name]
            single_reward = multi_reward[agent_name]
            single_done = multi_done[agent_name]
            agent.replay_buffer.add_memo(single_obs,single_next_obs, state, next_state, single_action, single_reward, single_done)
           
        # update brain every fixed steps
        multi_batch_obses = []
        multi_batch_next_obses = []
        multi_batch_states = []
        multi_batch_next_states = []
        multi_batch_actions = []
        multi_batch_next_actions = []
        multi_batch_online_actions = []
        multi_batch_rewards = []
        multi_batch_dones = []

        #sample
        current_memo_size = min(MEMORY_SIZE, total_step+1)
        if current_memo_size < BATCH_SIZE:
            batch_idx = range(0, current_memo_size)
        else:
            batch_idx = np.random.choice(current_memo_size, BATCH_SIZE)
        for agent_i in range(NUM_AGENT):
            agent = agents[agent_i]
            batch_obses, batch_next_obses, batch_states, batch_next_states, \
                batch_actions, batch_rewards, batch_dones = agent.replay_buffer.sample(batch_idx)
            
            #single 和 Batch
            batch_obses_tensor = torch.tensor(batch_obses, dtype=torch.float).to(device)
            batch_next_obses_tensor = torch.tensor(batch_next_obses, dtype=torch.float).to(device)
            batch_states_tensor = torch.tensor(batch_states, dtype=torch.float).to(device)
            batch_next_states_tensor = torch.tensor(batch_next_states, dtype=torch.float).to(device)
            batch_actions_tensor = torch.tensor(batch_actions, dtype=torch.float).to(device)
            batch_rewards_tensor = torch.tensor(batch_rewards, dtype=torch.float).to(device)
            batch_dones_tensor = torch.tensor(batch_dones, dtype=torch.float).to(device)

            #multi + batch
            multi_batch_obses.append(batch_obses_tensor)
            multi_batch_next_obses.append(batch_next_obses_tensor)
            multi_batch_states.append(batch_states_tensor)
            multi_batch_next_states.append(batch_next_states_tensor)
            multi_batch_actions.append(batch_actions_tensor)

            single_batch_next_action = agent.target_actor.forward(batch_next_obses_tensor)
            multi_batch_next_actions.append(single_batch_next_action)
            single_batch_online_actions = agent.actor.forward(batch_obses_tensor)
            multi_batch_online_actions.append(single_batch_online_actions)

            multi_batch_rewards.append(batch_rewards_tensor)
            multi_batch_dones.append(batch_dones_tensor)


        multi_batch_actions_tensor = torch.cat(multi_batch_actions, dim=1).to(device)
        multi_batch_next_actions_tensor = torch.cat(multi_batch_next_actions, dim=1).to(device)
        multi_batch_online_actions_tensor = torch.cat(multi_batch_online_actions, dim=1).to(device)

        # Update critic and actor
        if (total_step + 1) % TARGET_UPDATE_INTERVAL == 0:
            for agent_i in range(NUM_AGENT):
                agent = agents[agent_i]
                
                batch_obses_tensor = multi_batch_obses[agent_i]
                batch_states_tensor = multi_batch_states[agent_i]
                batch_next_states_tensor = multi_batch_next_states[agent_i]
                batch_actions_tensor = multi_batch_actions[agent_i]
                batch_rewards_tensor = multi_batch_rewards[agent_i]
                batch_dones_tensor = multi_batch_dones[agent_i]

                #target critic
                critic_target_q = agent.target_critic.forward(batch_next_states_tensor, 
                                                              multi_batch_next_actions_tensor.detach())
                y = (batch_rewards_tensor + (1 - batch_dones_tensor) * agent.gamma * critic_target_q).flatten()
                critic_q = agent.critic.forward(batch_states_tensor, multi_batch_actions_tensor).flatten()

                # update critic
                critic_loss = nn.MSELoss()(critic_q, y)
                agent.critic.optimizer.zero_grad()
                critic_loss.backward()
                
                # print("\n=== Critic Gradients ===")
                # for name, param in agent.critic.named_parameters():
                #     if param.grad is not None:
                #         print(f"{name}: mean={param.grad.mean().item():.6f}, max={param.grad.max().item():.6f}")
                #     else:
                #         print(f"{name}: No gradient")

                agent.critic.optimizer.step()
                # Update actor
                if total_step > WARMUP_EPISODES:
                    multi_batch_online_actions_list = [
                        single_batch_online_action if j == agent_i else single_batch_online_action.detach() for
                        j, single_batch_online_action in enumerate(multi_batch_online_actions)]

                    multi_batch_online_actions_tensor = torch.cat(multi_batch_online_actions_list, dim=1).to(device)


                    actor_loss = agent.critic.forward(batch_states_tensor,
                                                      multi_batch_online_actions_tensor).flatten()
                    actor_loss = -torch.mean(actor_loss)
                    agent.actor.optimizer.zero_grad()
                    actor_loss.backward(retain_graph=True)
                    #  # check grad（Actor）
                    # if (total_step + 1) % TARGET_UPDATE_INTERVAL % 100 == 0:
                    #     print(f"\n[Actor@Step {total_step}] Agent {agent_i}:")
                    #     for name, param in agent.actor.named_parameters():
                    #         if param.grad is not None:
                    #             print(f"  {name:25} mean={param.grad.mean().item():.3e} | max={param.grad.max().item():.3e}")
                    # agent.actor.optimizer.step()

                    #  Update target critic
                    for target_param, param in zip(agent.target_critic.parameters(), agent.critic.parameters()):
                        target_param.data.copy_(agent.tau * param.data + (1.0 - agent.tau) * target_param.data)

                    #  Update target actor
                    for target_param, param in zip(agent.target_actor.parameters(), agent.actor.parameters()):
                        target_param.data.copy_(agent.tau * param.data + (1.0 - agent.tau) * target_param.data)
        
        for agent_name in agent_name_list:
            current_episode_actions[agent_name].append(multi_action[agent_name].tolist())
            current_episode_obs[agent_name].append(multi_next_obs[agent_name].tolist())
        multi_obs = multi_next_obs
        episode_reward += sum([single_reward for single_reward in multi_reward.values()])
        # print(f"Episode reward: {episode_reward}")

    # Store the current episode data in the total record
    for agent_name in agent_name_list:
        agent_data[agent_name]['actions'][episode_i] = current_episode_actions[agent_name]
        agent_data[agent_name]['obs'][episode_i] = current_episode_obs[agent_name]
    EPISODE_REWARD_BUFFER.append(episode_reward)
    AVG_EPISODE_REWARD_BUFFER.append(np.mean(EPISODE_REWARD_BUFFER))
    print(f"Episode: {episode_i + 1} Reward: {np.round(episode_reward, 4)}, AVG Reward: {np.round(AVG_EPISODE_REWARD_BUFFER[-1], 4)}")


    # 3.save model
    if episode_i == 0:
        highest_avg_reward = AVG_EPISODE_REWARD_BUFFER[-1]
        highest_reward = EPISODE_REWARD_BUFFER[-1]
    if episode_i > WARMUP_EPISODES and AVG_EPISODE_REWARD_BUFFER[-1] > highest_avg_reward:
        highest_avg_reward = AVG_EPISODE_REWARD_BUFFER[-1]
        print(f"--------------Highest Average Reward Update: {round(AVG_EPISODE_REWARD_BUFFER[-1], 4)}--------------")
        # print(f"Saving model at episode {episode_i}")
        for agent_i in range(NUM_AGENT):
            agent = agents[agent_i]
            flag = os.path.exists(agent_path)
            if not flag:
                os.makedirs(agent_path)
            torch.save(agent.actor.state_dict(), f'{agent_path}' + f'agent_{agent_i}_actor_{scenario}_{timestamp}.pth')
    elif episode_i > WARMUP_EPISODES and EPISODE_REWARD_BUFFER[-1] > highest_reward:
        highest_reward = EPISODE_REWARD_BUFFER[-1]
        print(f"Highest reward updated: {round(EPISODE_REWARD_BUFFER[-1], 4)}")
        # print(f"Saving model at episode {episode_i}")
        for agent_i in range(NUM_AGENT):
            agent = agents[agent_i]
            flag = os.path.exists(agent_path)
            if not flag:
                os.makedirs(agent_path)
            torch.save(agent.actor.state_dict(), f'{agent_path}' + f'agent_{agent_i}_actor_{scenario}_{timestamp}.pth')
env.close()
# After training is completed, save it to Excel
with pd.ExcelWriter('agent_training_data.xlsx') as writer:
    for agent_name in agent_name_list:
        # action
        action_df = pd.DataFrame.from_dict(
            agent_data[agent_name]['actions'], 
            orient='index'
        )
        action_df.index.name = 'Episode'
        action_df.columns = [f'Step_{i}' for i in range(action_df.shape[1])]
        action_df.to_excel(writer, sheet_name=f'{agent_name}_actions')
        
        # obs
        obs_records = []
        for ep in range(NUM_EPISODE):
            for step in range(len(agent_data[agent_name]['obs'][ep])):
                obs_records.append({
                    'Episode': ep,
                    'Step': step,
                    **{f'obs_dim_{i}': val 
                       for i, val in enumerate(agent_data[agent_name]['obs'][ep][step])}
                })
        
        obs_df = pd.DataFrame(obs_records).set_index(['Episode', 'Step'])
        obs_df.to_excel(writer, sheet_name=f'{agent_name}_obs')

print("The training data has been saved to agent_training_data.xlsx")

Episode: 1 Reward: -727.1753, AVG Reward: -727.1753


C:\Users\Clouds\AppData\Local\Temp\ipykernel_20796\3200504151.py:195: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  net_load = float(current_loads[agent_id] - charge_power)  # 放电为正，充电为负
C:\Users\Clouds\AppData\Local\Temp\ipykernel_20796\3200504151.py:221: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  float(self.battery_soc[agent_id])


Episode: 2 Reward: -725.4301, AVG Reward: -726.3027
Episode: 3 Reward: -726.484, AVG Reward: -726.3631
Episode: 4 Reward: -727.3724, AVG Reward: -726.6154
Episode: 5 Reward: -726.5315, AVG Reward: -726.5986
Episode: 6 Reward: -727.0416, AVG Reward: -726.6725
Episode: 7 Reward: -727.1602, AVG Reward: -726.7421
Episode: 8 Reward: -726.7262, AVG Reward: -726.7401
Episode: 9 Reward: -727.2631, AVG Reward: -726.7983
Episode: 10 Reward: -727.4172, AVG Reward: -726.8601
Episode: 11 Reward: -727.3383, AVG Reward: -726.9036
Episode: 12 Reward: -726.8725, AVG Reward: -726.901


KeyboardInterrupt: 